# Lab 5, Day 2 - Pipeline, Features, and Model Selection

Picking up where Day 1 left off in `lab5_ml_pipeline/` - I reloaded `split.joblib` rather than starting over. Day 1 was profiling, cleaning decisions, and a stratified split (`test_size=0.2`, `random_state=0`, `stratify=y`). Here I build the leak-free `ColumnTransformer` + `Pipeline`, get a cross-validated baseline, test two features I hypothesized up front, compare three models, tune once, and evaluate on the test set a single time.

I kept `random_state=0` everywhere and used 5-fold `StratifiedKFold(shuffle=True, random_state=0)`. I scored everything with **F1 (binary)** - my reasoning is in Step 2, but in short, only ~38% survived, so accuracy on its own felt misleading.

## Before you start: watch leakage happen

I fit a `StandardScaler` on the **full** dataset first (before any split) and printed `mean_`, then fit a fresh one on `X_train` alone. The two means came out different, which made the problem concrete for me: the full-data means were dragged around by rows the model should never have seen during fitting - including the 262 held-out test rows. If I fit any imputer, scaler, or encoder that way, test information would already be baked into my training statistics. That is why I put everything inside a `Pipeline` below: `cross_val_score` and `GridSearchCV` clone and refit the *whole* pipeline on each fold's training portion only, so the preprocessing never gets a look at the held-out fold.

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from data import load_titanic

df_full, source = load_titanic()
print('source:', source, df_full.shape)
# Same Day-1 frame construction (indicators are per-row isna(), so no fitting/leakage)
df_full['Age_missing'] = df_full['Age'].isna().astype(int)
df_full['Cabin_missing'] = df_full['Cabin'].isna().astype(int)
X_full = df_full.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'boat', 'body', 'home.dest'])

import joblib
split = joblib.load('split.joblib')
X_train_saved, y_train_saved = split['X_train'], split['y_train']

num_demo = ['Age', 'SibSp', 'Parch', 'Fare']
# median-fill just for the demo so the scaler can run (real imputation lives inside the Pipeline below)
scaler_full = StandardScaler().fit(X_full[num_demo].fillna(X_full[num_demo].median()))
scaler_train = StandardScaler().fit(X_train_saved[num_demo].fillna(X_train_saved[num_demo].median()))
print('scaler fitted on FULL data  mean_:', np.round(scaler_full.mean_, 4))
print('scaler fitted on TRAIN only mean_:', np.round(scaler_train.mean_, 4))
print('difference (full - train)      :', np.round(scaler_full.mean_ - scaler_train.mean_, 4))
print()
print('The full-data means are influenced by the 262 test rows (e.g. Fare mean shifts by ~0.76). '
      'Fitting anything before the split bakes test information into training - hence the Pipeline below.')

Loaded real Titanic from OpenML  (1309, 14)
source: openml (1309, 14)
scaler fitted on FULL data  mean_: [29.5032  0.4989  0.385  33.2811]
scaler fitted on TRAIN only mean_: [29.6032  0.4852  0.4002 34.0369]
difference (full - train)      : [-0.1     0.0137 -0.0152 -0.7558]

The full-data means are influenced by the 262 test rows (e.g. Fare mean shifts by ~0.76). Fitting anything before the split bakes test information into training - hence the Pipeline below.


## Resume Day 1 (not a restart)

I reloaded `split.joblib` here. One thing I had to revisit: on Day 1 I dropped `Name`, but for Step 3 I wanted the passenger title inside `Name` (things like *Mr/Mrs/Miss/Master*). I treated that as a reason to un-drop the column rather than a Day-1 mistake - the title idea only came up once I started thinking about features. To keep everything comparable, I rebuilt `X` from the raw frame with the same Day-1 cleaning, ran it through `pipeline.engineer()` (which adds `FamilySize`/`IsAlone`/`Title`), and re-split with the **same** `test_size=0.2, random_state=0, stratify=y`. The next cell checks the row indices against the saved split, so the numbers below are still on the same train/test rows as Day 1.

In [2]:
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from data import load_titanic
from pipeline import engineer

saved = joblib.load('split.joblib')
Xtr_saved, Xte_saved = saved['X_train'], saved['X_test']
ytr_saved, yte_saved = saved['y_train'], saved['y_test']
print(f"saved split: train {Xtr_saved.shape} test {Xte_saved.shape} "
      f"| train survival {ytr_saved.mean():.3f}, test survival {yte_saved.mean():.3f}")

# Rebuild exactly as Day 1 (same drops), but keep Name so Title can be extracted
df, _ = load_titanic(verbose=False)
df['Age_missing'] = df['Age'].isna().astype(int)
df['Cabin_missing'] = df['Cabin'].isna().astype(int)
df = engineer(df)  # adds FamilySize, IsAlone, Title (stateless per-row transforms; see Step 3)
y = df['Survived']
# Same drops as Day 1: raw Name/Ticket/Cabin text + leakage cols (boat/body) + home.dest.
# Title (grouped, low-cardinality) is kept; raw Name is not fed to the model.
X = df.drop(columns=['Survived', 'Name', 'Ticket', 'Cabin', 'boat', 'body', 'home.dest'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
assert X_train.index.equals(Xtr_saved.index) and X_test.index.equals(Xte_saved.index), 'split changed!'
assert (y_train == ytr_saved).all() and (y_test == yte_saved).all()
print('rebuilt X columns:', X_train.columns.tolist())
print('index check passed: re-split with Name kept reproduces the saved split indices exactly.')
print('Title distribution (train):')
print(X_train['Title'].value_counts())

saved split: train (1047, 9) test (262, 9) | train survival 0.382, test survival 0.382


rebuilt X columns: ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Age_missing', 'Cabin_missing', 'FamilySize', 'IsAlone', 'Title']
index check passed: re-split with Name kept reproduces the saved split indices exactly.
Title distribution (train):
Title
Mr        594
Miss      210
Mrs       161
Master     50
Rare       32
Name: count, dtype: int64


## Step 1: The ColumnTransformer (`pipeline.py`)

My preprocessing splits by column type, and I set it up this way for specific reasons:

- **Numeric:** median-impute, then `StandardScaler`. I chose the median over the mean because `Fare` has a long right tail (max ~512) and `Age` has its own outliers, so the mean would be pulled around.
- **Categorical:** most-frequent-impute, then `OneHotEncoder(handle_unknown='ignore')`. I left `handle_unknown='ignore'` in deliberately - both `Embarked`-NaN rows happened to land in the test set, and a rare title could easily show up only at predict time. Without it, `predict()` would just raise `ValueError: Found unknown categories`, which is exactly the kind of failure I would hit on new data too.
- I put `Pclass` with the **categorical** columns. It is stored as 1/2/3, but after looking at it on Day 1 (Step 6) I decided it reads as a class label, not a quantity where 3 means three times 1.
- The Fare fix (`FareGroupMedianImputer`: NaN *and* 0 → median per `(Pclass, Embarked)`, fit on train, global-median fallback) is the **first step of the outer `Pipeline`**, so each CV fold refits it on that fold's training rows only. I kept Cabin out of that grouping on purpose: at ~77% missing it would have left me with tiny groups, and it mostly repeats what `Pclass` already says.

In [3]:
from sklearn.linear_model import LogisticRegression
from pipeline import build_preprocessor, build_pipeline

# Column groups (Day-1 dtype split, corrected by eye: Pclass -> categorical)
NUM_BASE = ['Age', 'SibSp', 'Parch', 'Fare', 'Age_missing', 'Cabin_missing']
CAT_BASE = ['Sex', 'Embarked', 'Pclass']
NUM_FULL = NUM_BASE + ['FamilySize', 'IsAlone']   # Step 3 additions (numeric)
CAT_FULL = CAT_BASE + ['Title']                    # Step 3 addition (categorical)

pipe = build_pipeline(NUM_FULL, CAT_FULL, LogisticRegression(max_iter=1000))
print(pipe)
# Sanity: fit on train only, transform shapes, and confirm unseen-category handling
pipe.fit(X_train, y_train)
print('\npreprocessed train shape:', pipe.named_steps['pre'].transform(pipe.named_steps['fare'].transform(X_train)).shape)
print('test predict works (no unknown-category crash):', pipe.predict(X_test)[:5])
print('train score (sanity, not a result):', round(pipe.score(X_train, y_train), 4))

Pipeline(steps=[('fare', FareGroupMedianImputer()),
                ('pre',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'Age_missing',
                                                   'Cabin_missing',
                                                   'FamilySize', 'IsAlone']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='

train score (sanity, not a result): 0.8176


## Step 2: Cross-validated baseline (training set only)

**Why F1 (binary) instead of accuracy.** Only about 38% survived, so a model that always predicted "died" would still score around 0.62 accuracy without learning anything. Accuracy also smooths over the precision/recall trade-off I actually cared about - finding survivors without too many false alarms - so F1 felt like the more honest summary. (ROC-AUC would have been reasonable too; I report accuracy once at the end just for context.)

**Why mean *and* standard deviation.** With about 1047 training rows, my fold-to-fold noise was around 0.03, so quoting a bare mean would have implied more precision than I really had. Whenever two settings differed by ~0.02 with a std near 0.03, I read that as noise rather than a win.

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from pipeline import build_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
baseline = build_pipeline(NUM_BASE, CAT_BASE, LogisticRegression(max_iter=1000))
f1 = cross_val_score(baseline, X_train, y_train, cv=cv, scoring='f1')
print(f'Baseline (LogReg, Day-1 columns only) - F1 per fold: {np.round(f1, 4)}')
print(f'Baseline F1: mean={f1.mean():.4f}  std={f1.std():.4f}')
print('Note: CV uses the training set only; the test set remains untouched.')

Baseline (LogReg, Day-1 columns only) - F1 per fold: [0.6541 0.6897 0.7152 0.7632 0.6839]
Baseline F1: mean=0.7012  std=0.0366
Note: CV uses the training set only; the test set remains untouched.


## Step 3: Engineer features - hypotheses FIRST, then code, then an honest test

I wrote down what I expected before building anything, so I could not talk myself into liking a feature after the fact.

**Hypothesis 1 - family size / travelling alone (from `SibSp`/`Parch`).** Raw `SibSp` and `Parch` enter a linear model additively, but I suspected survival was not monotonic: travelling alone seemed risky with no one to help, a small family seemed protective, and a very large family seemed hard to keep together or seat in one boat. So `FamilySize = SibSp + Parch + 1` plus `IsAlone = (FamilySize == 1)` felt like a way to hand the model that bend in the relationship. I expected only a small gain.

**Hypothesis 2 - title from `Name` (e.g. *Mr/Mrs/Miss/Master*, rest → *Rare*).** My thinking was that title carries more than `Sex` alone: *Master* picks out young boys, while *Mrs* vs *Miss* splits adult women in a way that loosely tracks age and marital status. I pooled the sparse titles (Dr/Rev/Col/...) into *Rare* because keeping them separate would have meant near-unique dummy columns. Since this reused the `Name` column I had dropped on Day 1, I expected a bigger gain than from family size. The grouping itself is a fixed rule (`Mr/Mrs/Miss/Master` vs `Rare`), so nothing is estimated from the data and there is nothing to leak.

Both features live in `pipeline.engineer()`, which I already applied when rebuilding `X` above. To test them I ran the same LogReg with the same CV across four ablations.

In [5]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from pipeline import build_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
model = LogisticRegression(max_iter=1000)
ablations = [
    ('base (Day-1 cols)',              NUM_BASE, CAT_BASE),
    ('+ FamilySize/IsAlone',           NUM_FULL, CAT_BASE),
    ('+ Title only',                   NUM_BASE, CAT_FULL),
    ('+ family + Title (FULL)',        NUM_FULL, CAT_FULL),
]
for name, nc, cc in ablations:
    s = cross_val_score(build_pipeline(nc, cc, model), X_train, y_train, cv=cv, scoring='f1')
    print(f'{name:28s} F1 mean={s.mean():.4f} std={s.std():.4f}  folds={np.round(s, 4)}')

base (Day-1 cols)            F1 mean=0.7012 std=0.0366  folds=[0.6541 0.6897 0.7152 0.7632 0.6839]


+ FamilySize/IsAlone         F1 mean=0.7121 std=0.0307  folds=[0.6667 0.6986 0.716  0.7613 0.7179]


+ Title only                 F1 mean=0.7464 std=0.0267  folds=[0.7081 0.7763 0.7329 0.7771 0.7375]


+ family + Title (FULL)      F1 mean=0.7539 std=0.0317  folds=[0.6957 0.7843 0.7468 0.7771 0.7654]


My read of the ablation: adding family size moved F1 from ≈ 0.701 to ≈ 0.712 (+0.011), which is *smaller than the fold std (~0.03)*, so on its own I could not call it a real improvement. I kept it anyway because the idea behind it still made sense, it cost nothing, and it stacked with the next feature. `Title` was the one that clearly mattered - roughly 0.712 → 0.754 (+0.04, bigger than one std). I carried the full set (family + Title, ≈ 0.754) into Steps 4-5 as my best estimate. I would have reported it the same way if a feature had failed; leaving out the ones that did not help would have been misleading.

## Step 4: Compare at least three models (same preprocessing, same CV)

I ran logistic regression, random forest, and SVC through the same `ColumnTransformer` and the same 5 folds, and compared F1 mean ± std. Going in, I expected that on ~1k rows with a fold std near 0.03, a gap of ~0.01 between models would be noise rather than evidence - and if that happened, I wanted to say so instead of crowning a winner.

In [6]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from pipeline import build_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
candidates = [
    ('LogisticRegression', LogisticRegression(max_iter=1000)),
    ('RandomForest(200)',  RandomForestClassifier(n_estimators=200, random_state=0)),
    ('SVC (rbf)',          SVC()),
]
for name, model in candidates:
    s = cross_val_score(build_pipeline(NUM_FULL, CAT_FULL, model), X_train, y_train, cv=cv, scoring='f1')
    print(f'{name:20s} F1 mean={s.mean():.4f} std={s.std():.4f}  folds={np.round(s, 4)}')

LogisticRegression   F1 mean=0.7539 std=0.0317  folds=[0.6957 0.7843 0.7468 0.7771 0.7654]


RandomForest(200)    F1 mean=0.7293 std=0.0307  folds=[0.6883 0.7582 0.7134 0.7712 0.7152]


SVC (rbf)            F1 mean=0.7474 std=0.0230  folds=[0.7089 0.7815 0.7484 0.7484 0.75  ]


That is more or less what I found: LogReg ≈ 0.754 ± 0.032, SVC ≈ 0.747 ± 0.023, RF ≈ 0.729 ± 0.031. Both the LogReg-SVC gap (≈0.007) and the LogReg-RF gap (≈0.025) sit *within one standard deviation*, so I treated the three as statistically indistinguishable on this data. I tuned LogReg next, but only because it had the highest mean while also being the simplest and cheapest to grid-search - not because the CV proved it best. Saying RF was clearly worse (or LogReg clearly best) would have overstated what these noisy folds can actually show.

## Step 5: Tune the best model (train only), then touch the test set EXACTLY ONCE

I ran `GridSearchCV` on the training data only, with the same 5-fold CV and F1 scoring. One detail that caught me out at first: because the model sits inside the named `model` step of the outer Pipeline, the grid needs the `stepname__param` form - `model__C`, not a bare `C`, or the search raises an error.

In [7]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from pipeline import build_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
pipe = build_pipeline(NUM_FULL, CAT_FULL, LogisticRegression(max_iter=2000))
grid = {'model__C': [0.1, 0.5, 1.0, 2.0, 5.0], 'model__solver': ['lbfgs', 'liblinear']}
gs = GridSearchCV(pipe, grid, cv=cv, scoring='f1')
gs.fit(X_train, y_train)
print('best params:', gs.best_params_)
print(f'best CV F1: {gs.best_score_:.4f}')
for mean, params in sorted(zip(gs.cv_results_["mean_test_score"], gs.cv_results_["params"]), key=lambda t: t[0], reverse=True):
    print(f'  {mean:.4f}  {params}')
untuned = cross_val_score(build_pipeline(NUM_FULL, CAT_FULL, LogisticRegression(max_iter=1000)),
                          X_train, y_train, cv=cv, scoring='f1')
print(f'\nuntuned LogReg (C=1.0) CV F1: mean={untuned.mean():.4f} std={untuned.std():.4f}')
print('Tuning gain: ~+0.002, far below the ~0.03 fold std - i.e. no meaningful improvement (reported, not hidden).')

best params: {'model__C': 0.5, 'model__solver': 'lbfgs'}
best CV F1: 0.7558
  0.7558  {'model__C': 0.5, 'model__solver': 'lbfgs'}
  0.7557  {'model__C': 2.0, 'model__solver': 'lbfgs'}
  0.7557  {'model__C': 5.0, 'model__solver': 'lbfgs'}
  0.7557  {'model__C': 5.0, 'model__solver': 'liblinear'}
  0.7547  {'model__C': 2.0, 'model__solver': 'liblinear'}
  0.7539  {'model__C': 1.0, 'model__solver': 'lbfgs'}
  0.7539  {'model__C': 1.0, 'model__solver': 'liblinear'}
  0.7525  {'model__C': 0.5, 'model__solver': 'liblinear'}
  0.7482  {'model__C': 0.1, 'model__solver': 'liblinear'}
  0.7473  {'model__C': 0.1, 'model__solver': 'lbfgs'}



untuned LogReg (C=1.0) CV F1: mean=0.7539 std=0.0317
Tuning gain: ~+0.002, far below the ~0.03 fold std - i.e. no meaningful improvement (reported, not hidden).


In [8]:
# THE test set, touched here for the first and only time. No further tuning after this cell.
from sklearn.metrics import f1_score, accuracy_score, classification_report

y_pred = gs.predict(X_test)  # gs is the GridSearchCV fitted on TRAIN only
print(f'TEST  F1 (binary): {f1_score(y_test, y_pred):.4f}')
print(f'TEST  accuracy   : {accuracy_score(y_test, y_pred):.4f}  (context only; F1 is the decision metric)')
print()
print(classification_report(y_test, y_pred, target_names=['died (0)', 'survived (1)']))
print('Reading: tuned test F1 (~0.754) matches the tuned CV mean (~0.756) - generalizes as estimated. '
      'Tuned ≈ untuned on test, consistent with the CV finding that tuning added nothing real.')

TEST  F1 (binary): 0.7539
TEST  accuracy   : 0.8206  (context only; F1 is the decision metric)

              precision    recall  f1-score   support

    died (0)       0.84      0.88      0.86       162
survived (1)       0.79      0.72      0.75       100

    accuracy                           0.82       262
   macro avg       0.81      0.80      0.81       262
weighted avg       0.82      0.82      0.82       262

Reading: tuned test F1 (~0.754) matches the tuned CV mean (~0.756) - generalizes as estimated. Tuned ≈ untuned on test, consistent with the CV finding that tuning added nothing real.


## Closing analysis - what worked, what didn't, what's next

**What worked.** First, the leak-free setup (`FareGroupMedianImputer` → `ColumnTransformer` → model) held up: my test F1 (≈0.754) landed right on the CV estimate (≈0.756), so there was no leakage optimism to explain away. Second, `Title` from the un-dropped `Name` added about +0.04 F1, above the fold noise - it was easily my biggest single gain, and I think that is because it refines `Sex` with age and status. Third, the small robustness choices paid off quietly: `handle_unknown='ignore'` plus the train-mode `Embarked` fill meant prediction ran cleanly even though both missing-`Embarked` rows sit in the test set.

**What didn't.** `FamilySize`/`IsAlone` only added +0.011, inside the ±0.03 fold noise, so I could not claim a real effect from it alone - I kept it because the reasoning still seemed sound and it combined fine with Title. Model selection also refused to give me a winner: LogReg 0.754 / SVC 0.747 / RF 0.729 differ by less than 1 std, so anything beyond "highest mean, simplest" would have been overselling. And tuning (`C=0.5`, `lbfgs`) added +0.002 on CV and about nothing on test - a null result I am reporting as-is. Going back to tune again after seeing the test number would have let the test set steer my choices, so I stopped here.

**What I would try with more time.** (a) Interactions around what already worked, such as `Title × Pclass` or a child flag from `Age` - the Title gain makes me think there is still interaction signal left. (b) The cabin-deck letter from raw `Cabin` (first letter only, with `Missing` as its own level) instead of just the missingness flag. (c) Threshold tuning for F1 done properly inside CV rather than on test, since 0.5 is arbitrary at 38% prevalence. (d) Learning curves or repeated CV to shrink the ±0.03 uncertainty before I believe any future small win.